# ___`hOUwie` in parallel___
--------------------------

In [1]:
import subprocess

In [ ]:
#-----------------
# CONSTANTS
#-----------------

R_FULL_PATH = r"C:/R-4.5.2/bin/R.exe"
RSCRIPT_FULL_PATH = r"C:/R-4.5.2/bin/Rscript.exe"

CONTINUOUS_TRAIT_MODELS = ("OUM", "OUMA", "OUMV", "OUMVA")
DISCRETE_TRAIT_MODELS = ("ER", "SYM", "ARD")

In [38]:
#-------------------
# TEMPLATES
#-------------------

def name_houwie_for_saving(discrete_model: str, continuous_model: str, null_model: bool, continuous_trait: str, extra_suffix: str) -> str:
    return f"{discrete_mode}_{continuous_model}_{continuous_trait}_{"CID" if null_model else "CD"}_{extra_suffix}.Rds"
    

def generate_houwie_rscript(phylogeny: str, traitdata: str, rate_cat: int, discrete_model: str, continuous_model: str, null_model: bool, savedir: str, nsims: int = 30) -> str:
    """
    whips up a (string format) R script on the fly, so it can be passed via the expression (-e) argument to R.exe or Rscript.exe
    """
    
    PSEUDO_R_SCRIPT_TEMPLATE = \
    r"""
    suppressPackageStartupMessages({{
        library("ape")
        library("phytools")
        library("corHMM")
        library("OUwie")
    }})
    
    phylogeny <- ape::read.tree("{}")
    data <- read.csv("{}")
    stopifnot(all(phylogeny$tip.label == data$binominal))
    
    model <- OUwie::hOUwie(phy = phylogeny, data = data, rate.cat = {}, discrete_model = "{}", continuous_model = "{}", nSim = {}, null.model = {})
    saveRDS(object = model, file = "{}")
    """

    # 1st placeholder - path to the phylogenetic tree
    # 2nd placeholder - path to the trait data (MUST BE NAME MATCHED TO THE PHYLOGENY)
    # 3rd placeholder - rate category (1, 2)
    # 4th placeholder - one of the discrete model types
    # 5th placeholder - one of the continuous model types
    # 6th placeholder - number of simulations to run
    # 7th placeholder - NULL model (TRUE or FALSE)
    # 8th placeholder - path to serialize the fit model
    
    # the trait data is expected to have the following three columns (in the specified order) - binominal names, discrete trait and continuous trait
    # the binominal names must be identical to the tip labels of the phylogeny

    return PSEUDO_R_SCRIPT_TEMPLATE.format(phylogeny, traitdata, rate_cat, discrete_model, continuous_model, nsims, "TRUE" if null_model else "FALSE",
                                           savedir + name_houwie_for_saving(discrete_model=discrete_model, continuous_model=continuous_model, null_model=null_model, ))
    

In [37]:
print(generate_houwie_rscript(phylogeny=r"../data/chapter2/uphylomaker/FRED_subset_collab_395sp.tre", traitdata=r"../data/chapter2/FREDv3subset/collab_ord1_SRL_RD_species_avgd.csv",
                       rate_cat=1, discrete_mode="OUM", continuous_model="ARD", null_model=False, savepath="./rdata/OUMARD.Rds", nsims=100))


    suppressPackageStartupMessages({
        library("ape")
        library("phytools")
        library("corHMM")
        library("OUwie")
    })

    phylogeny <- ape::read.tree("../data/chapter2/uphylomaker/FRED_subset_collab_395sp.tre")
    data <- read.csv("../data/chapter2/FREDv3subset/collab_ord1_SRL_RD_species_avgd.csv")
    stopifnot(all(phylogeny$tip.label == data$binominal))

    model <- OUwie::hOUwie(phy = phylogeny, data = data, rate.cat = 1, discrete_model = "OUM", continuous_model = "ARD", nSim = 100, null.model = FALSE)
    saveRDS(object = model, file = "./rdata/OUMARD.Rds")
    


In [9]:
subprocess.run([r"C:\Program Files\Python313\python.exe",  "--version"], shell=False, capture_output=True, text=True)

CompletedProcess(args=['C:\\Program Files\\Python313\\python.exe', '--version'], returncode=0, stdout='Python 3.13.11\n', stderr='')